# Stage 4: Plots

Generates all final figures per condition (Nyquist, Bode, DRT stacked, Arrhenius)
and a multi-condition Brouwer p(O₂) diagram aggregating all atmospheric conditions.

**Reads:** `{sample_id}/Results/{condition}/stage3_drt.xlsx` · `stage3_fit.xlsx` · `ISM validation/{condition}/*.ism`
**Writes:** `{sample_id}/Results/{condition}/{DRT,Nyquist-Bode,Arrhenius}/` · `Results/pO2/`

## Quick links
- [Configuration](#configuration): `SAMPLE_ID`, `CONDITION_FILTER`, `L_m`, `D_m`, `PLOT_WINDOWS`, `DRT_TAU_MAX`
- [Step 1: Figures per condition](#step-1-figures-per-condition): DRT stacked, Nyquist, Bode, Arrhenius
- [Step 2: Brouwer p(O₂)](#step-2-brouwer-po-all-conditions): multi-condition σ vs p(O₂) diagram
- [Step 2b: Brouwer peak selector](#step-2b-brouwer-select-peak-and-temperatures): interactive peak and temperature filter
- [Step 3: transference numbers](#step-3-ionic-electronic-decomposition-transference-numbers): per-process σ_ion/σ_el split from the Brouwer fit
- [Output summary](#output-summary): exported figure paths

**Prerequisite:** run `stage3_drt.ipynb` for this sample first: it stores the pellet geometry (`L_m`, `D_m`) in session.json and writes the `stage3_fit.xlsx` / `stage3_drt.xlsx` files Stage 4 reads.

**Workflow:** edit the config cell → run the plot cell → use the Step 2b Brouwer peak selector without scrolling. Per-(condition, T) axis crops can be stored in `PLOT_WINDOWS` (session.json → stage4_params).

In [ ]:
import json
import sys
from pathlib import Path
from pipeline.interactive import select_sample
from pipeline.session import load_sample, update_sample

NOTEBOOK_DIR = Path.cwd()

sample_id = select_sample(NOTEBOOK_DIR, show_list=True)

_cfg = load_sample(sample_id)

def _update_session(**fields):
    update_sample(sample_id, **fields)

# Optional: process only specific conditions (leave empty [] to process ALL)
CONDITION_FILTER = []
condition_filter = CONDITION_FILTER

# Sample geometry - loaded from session.json (written by Stage 3)
L_m = _cfg.get("L_m")
D_m = _cfg.get("D_m")

if L_m is None or D_m is None:
    raise ValueError(
        f"Sample geometry (L_m, D_m) not found in session.json for '{sample_id}'.\n"
        "Run stage3_drt.ipynb first: its configuration cell asks for pellet "
        "thickness L and diameter D and saves them for Stage 4."
    )

# USE_SAVED_PARAMS: True resumes the calibration saved in session.json
# (normal use). False writes the values in THIS cell to session.json:
# edit below, set False, run once, then set back to True.
USE_SAVED_PARAMS = False

_p4 = _cfg.get("stage4_params", {}) if USE_SAVED_PARAMS else {}
print("Stage 4 parameter source:",
      "session.json (saved)" if _p4 else "notebook values (will overwrite session.json)")

DRT_TAU_MAX      = _p4.get("DRT_TAU_MAX",      0.1)
BROUWER_PEAK_ID  = _p4.get("BROUWER_PEAK_ID",  1)
BROUWER_TEMPS    = _p4.get("BROUWER_TEMPS",     None)
# Reference slope guides drawn on the Brouwer diagrams; any subset of
# "-1/4", "-1/6", "0", "+1/6", "+1/4"
BROUWER_SLOPES   = _p4.get("BROUWER_SLOPES",   ["-1/4", "-1/6", "0", "+1/6", "+1/4"])
# Exclude T below this value [°C] from all Arrhenius fits (None = use all):
# set it when peak identity is not resolved at low T (e.g. N_peaks drops to 3)
ARRHENIUS_T_MIN  = _p4.get("ARRHENIUS_T_MIN",   None)
# Peaks whose series sum forms the HF block in the single-panel sigma
# Arrhenius (e.g. [1, 2]); the sum is drawn over the full T range while
# the separated branches respect ARRHENIUS_T_MIN. None disables the figure.
ARRHENIUS_SUM_PEAKS = _p4.get("ARRHENIUS_SUM_PEAKS", None)
# Brouwer exponent x for the ionic/electronic decomposition (0.25 in the
# dilute defect regime; 1/6 in other regimes)
TRANSF_EXPONENT  = _p4.get("TRANSF_EXPONENT",   1/4)
# Peaks shown in the transference figures (None = all). The table always
# covers every peak; restrict the FIGURES to transport processes (e.g. [1, 2])
# because t_ion is physically meaningful only for bulk/GB, not electrodes.
TRANSF_PEAK_IDS  = _p4.get("TRANSF_PEAK_IDS", [1,2])
# session.json stringifies dict keys: restore per-T int keys, keep
# condition-level string keys (z_max, freq_min, ...) as-is.
PLOT_WINDOWS = {
    cond: {(int(k) if isinstance(k, str) and k.isdigit() else k): v
           for k, v in win.items()}
    for cond, win in _p4.get("PLOT_WINDOWS", {}).items()
}

def _stage4_params() -> dict:
    return {
        "DRT_TAU_MAX":      DRT_TAU_MAX,
        "BROUWER_PEAK_ID":  BROUWER_PEAK_ID,
        "BROUWER_TEMPS":    BROUWER_TEMPS,
        "BROUWER_SLOPES":   BROUWER_SLOPES,
        "ARRHENIUS_T_MIN":  ARRHENIUS_T_MIN,
        "ARRHENIUS_SUM_PEAKS": ARRHENIUS_SUM_PEAKS,
        "TRANSF_EXPONENT":  TRANSF_EXPONENT,
        "TRANSF_PEAK_IDS":  TRANSF_PEAK_IDS,
        "PLOT_WINDOWS":     PLOT_WINDOWS,
    }

_update_session(stage4_params=_stage4_params())


def _nyquist_xylim(window: dict) -> tuple:
    xlim = ylim = None
    z_min = window.get("z_min", 0)
    if "z_max" in window:
        xlim = (z_min, window["z_max"])
        ylim = (0, window["z_max"])
    return xlim, ylim


def _bode_freqlim(window: dict) -> tuple | None:
    fmin = window.get("freq_min")
    fmax = window.get("freq_max")
    if fmin is None and fmax is None:
        return None
    return (fmin if fmin is not None else 1e-3, fmax if fmax is not None else 1e9)

In [ ]:
# inline backend: more reliable than ipympl with ipywidgets panels.
get_ipython().run_line_magic("matplotlib", "inline")  # type: ignore[name-defined]

import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from pipeline.ingest import load_ism, load_csv_spectrum
from pipeline.drt import clip_spectrum
from pipeline.quality import strip_inductive
from pipeline.plots import (
    apply_pub_style,
    plot_drt_stacked,
    plot_nyquist_multipanel,
    plot_bode,
    plot_arrhenius_panel,
    plot_arrhenius_sigma,
    plot_brouwer,
    build_arrhenius_results,
)

# Apply publication style once
apply_pub_style()

sample_dir   = NOTEBOOK_DIR / sample_id
RESULTS_BASE = sample_dir / "Results"

# Collect conditions that have completed Stage 3
all_conditions = sorted([
    d.name for d in RESULTS_BASE.iterdir()
    if d.is_dir()
    and (d / "stage3_fit.xlsx").exists()
    and (d / "stage3_drt.xlsx").exists()
])

conditions = (
    [c for c in all_conditions if c in condition_filter]
    if condition_filter else all_conditions
)

print(f"Sample     : {sample_id}")
print(f"Geometry   : L = {L_m*1e3:.3f} mm  D = {D_m*1e3:.3f} mm")
print(f"Conditions with Stage 3 output ({len(conditions)}):")
for c in conditions:
    print(f"  {c}")


def _condition_window(condition: str) -> dict:
    """Condition-level PLOT_WINDOWS entry: only top-level string keys (skip per-T int keys)."""
    cw = PLOT_WINDOWS.get(condition, {})
    return {k: v for k, v in cw.items() if isinstance(k, str)}
try:
    import ipywidgets as W
    from IPython.display import display as _display, clear_output as _clear
    _HAS_WIDGETS = True
except Exception as _exc:
    print(f"[INFO] ipywidgets not installed ({_exc}); control panels disabled.")
    _HAS_WIDGETS = False


## Step 1: Figures per condition

For each condition: loads the validated EIS spectra and the Zarc parameters from Stage 3,
then produces DRT stacked, Nyquist, Bode and Arrhenius figures. All fitted peaks appear in the Arrhenius panels; the R²(τ) of each fit is reported in the summary table.

All figures are saved automatically to `Results/{condition}/` as PNG + PDF.
*Arrhenius panels: `Ea_cond` from ln(σT) vs 1/T (long-range transport), `Ea_pol` from ln(τ) (local relaxation; not expected to be linear for electrode processes), `Ea_C` from ln(C_eff) (net: Ea_pol − Ea_cond). Derivations: README → Physical formulae.*


In [ ]:
# Collect all peak data across conditions (used later for the Brouwer diagram)
all_peaks_df_list = []

# Cache of in-memory data per condition; used by the live control panel below
_plot_cache: dict[str, dict] = {}

for condition in conditions:
    print(f"\n{'='*70}")
    print(f"Condition: {condition}")
    print(f"{'='*70}")

    res_dir  = RESULTS_BASE / condition
    val_dir  = sample_dir / "ISM validation" / condition
    csv_dir  = sample_dir / "input_spectra" / condition   # CSV entry mode

    # Load Stage 3 outputs
    df_fit_peaks   = pd.read_excel(res_dir / "stage3_fit.xlsx",  sheet_name="Peaks")
    df_fit_summary = pd.read_excel(res_dir / "stage3_fit.xlsx",  sheet_name="Summary")
    df_drt_spectra = pd.read_excel(res_dir / "stage3_drt.xlsx",  sheet_name="DRT_Spectra")
    df_kk_sel      = pd.read_excel(res_dir / "stage2_kk.xlsx",   sheet_name="Selected")

    # Collect for Brouwer aggregation
    all_peaks_df_list.append(df_fit_peaks)

    temps_available = sorted(df_fit_summary["T_nominal"].unique())
    print(f"  Temperatures: {[int(t) for t in temps_available]}")

    # Load ISM data (validated, same files used in Stage 3)
    records    = {}   # {T_nominal: (freq, Z_re, Z_im)}
    fit_params = {}   # {T_nominal: {R0, R, tau, alpha}}

    for _, row in df_kk_sel.iterrows():
        T_nom  = int(row["T_nominal"])
        fname  = row["file"]
        f_min  = row["f_min_cut"] if pd.notna(row.get("f_min_cut")) else None
        f_max  = row["f_max_cut"] if pd.notna(row.get("f_max_cut")) else None

        ism_path = val_dir / fname
        if not ism_path.exists() and (csv_dir / fname).exists():
            ism_path = csv_dir / fname
        if not ism_path.exists():
            print(f"  [WARN] Not found: {fname}; skipping T={T_nom}")
            continue

        rec  = (load_csv_spectrum(ism_path)
                if ism_path.suffix.lower() in (".csv", ".txt")
                else load_ism(ism_path))
        # Figures show only physically valid points: the same Z' >= 0 / Z'' >= 0
        # criterion applied before Lin-KK (no passive circuit can produce them;
        # verified not to affect the fitted parameters)
        _fs, _zrs, _zis, _n_str = strip_inductive(rec.freq, rec.Z_re, rec.Z_im)
        freq, Z_re, Z_im = clip_spectrum(_fs, _zrs, _zis, f_min, f_max)
        records[T_nom] = (freq, Z_re, Z_im)

        sub = df_fit_peaks[df_fit_peaks["T_nominal"] == T_nom]
        if not sub.empty:
            sum_row = df_fit_summary[df_fit_summary["T_nominal"] == T_nom]
            R0_val  = float(sum_row["R0"].iloc[0]) if not sum_row.empty else None
            fit_params[T_nom] = {
                "R0":    R0_val,
                "R":     sub["R_i"].values.tolist(),
                "tau":   sub["tau_i"].values.tolist(),
                "alpha": sub["alpha_i"].values.tolist(),
            }

    _plot_cache[condition] = {
        "records":        records,
        "fit_params":     fit_params,
        "df_drt_spectra": df_drt_spectra,
        "df_fit_peaks":   df_fit_peaks,
        "df_fit_summary": df_fit_summary,
        "res_dir":        res_dir,
    }

    # Output sub-directories
    drt_dir  = res_dir / "DRT"
    nq_dir   = res_dir / "Nyquist-Bode"
    arr_dir  = res_dir / "Arrhenius"

    # Resolve condition-level plot window (None = full range)
    cw = _condition_window(condition)
    nyq_xlim, nyq_ylim = _nyquist_xylim(cw)
    bode_freq          = _bode_freqlim(cw)

    # 1. DRT stacked plot
    print("  → DRT stacked plot")
    if not df_drt_spectra.empty:
        fig_drt = plot_drt_stacked(
            df_spectra   = df_drt_spectra,
            condition    = condition,
            save_dir     = drt_dir,
            tau_max      = DRT_TAU_MAX,
        )
        plt.show()
        plt.close(fig_drt)
    else:
        print("    [SKIP] No DRT spectra data.")

    # 2. Nyquist overlay (with optional condition-level crop)
    print("  → Nyquist overlay" + (f"  [window z_max={cw.get('z_max')} kΩ]" if "z_max" in cw else ""))
    if records:
        fig_nq = plot_nyquist_multipanel(
            records    = records,
            fit_params = fit_params,
            condition  = condition,
            save_dir   = nq_dir,
            xlim       = nyq_xlim,
            ylim       = nyq_ylim,
        )
        plt.show()
        plt.close(fig_nq)

    # 3. Bode plot (with optional condition-level freq window)
    print("  → Bode plot" + (f"  [freq window {bode_freq}]" if bode_freq else ""))
    if records:
        fig_bode = plot_bode(
            records    = records,
            fit_params = fit_params,
            condition  = condition,
            save_dir   = nq_dir,
            freq_lim   = bode_freq,
        )
        plt.show()
        plt.close(fig_bode)

    # 4. Arrhenius 2×2 panel
    print("  → Arrhenius panel")
    if not df_fit_peaks.empty:
        fig_arr, results_all = plot_arrhenius_panel(
            df_peaks     = df_fit_peaks,
            L_m          = L_m,
            D_m          = D_m,
            condition    = condition,
            save_dir     = arr_dir,
            t_min        = ARRHENIUS_T_MIN,
        )
        plt.show()
        plt.close(fig_arr)

        # Print activation energy summary table
        print(f"\n  Activation energies; {condition}")
        print(f"  {'Peak':<10} {'Ea_cond (eV)':<18} {'Ea_pol (eV)':<18} "
              f"{'Ea_C (eV)':<18} {'R2_cond':<10} {'R2_pol':<10} {'R2_C':<10}")
        print(f"  {'-'*94}")
        for r in results_all:
            def _fmt(v, e):
                return f"{v:.3f}±{e:.3f}" if not (np.isnan(v) or np.isnan(e)) else "N/A"
            def _r2(v):
                return f"{v:.4f}" if not np.isnan(v) else "N/A"
            print(f"  {r['Peak']:<10} {_fmt(r['Ea_cond'], r['Ea_cond_err']):<18} "
                  f"{_fmt(r['Ea_pol'], r['Ea_pol_err']):<18} "
                  f"{_fmt(r['Ea_C'], r['Ea_C_err']):<18} "
                  f"{_r2(r['R2_cond']):<10} {_r2(r['R2_pol']):<10} {_r2(r['R2_C']):<10}")
        print()

    # 5. HF-block sigma Arrhenius: separated branches + series sum
    if ARRHENIUS_SUM_PEAKS and not df_fit_peaks.empty:
        print("  → HF-block sigma Arrhenius")
        fig_sig = plot_arrhenius_sigma(
            df_peaks     = df_fit_peaks,
            L_m          = L_m,
            D_m          = D_m,
            condition    = condition,
            save_dir     = arr_dir,
            t_min        = ARRHENIUS_T_MIN,
            sum_peak_ids = ARRHENIUS_SUM_PEAKS,
        )
        if fig_sig is not None:
            plt.show()
            plt.close(fig_sig)

    print(f"  Figures saved in {res_dir.relative_to(NOTEBOOK_DIR)}")

print(f"\n{'='*70}")
print(f"Per-condition figures complete; {len(conditions)} condition(s).")

## Step 2: Brouwer p(O₂); all conditions

Aggregates data for peak `BROUWER_PEAK_ID` from **all** conditions and plots log₁₀(σ) vs log₁₀(p(O₂)).

Each symbol encodes a temperature (400–600 °C). Slope guides −¼, plateau, +¼ for interpretation.

> **Note**: the Brouwer diagram is physically meaningful only if the same `peak_id` represents the same process across all conditions. Verify using the C_eff magnitude and Arrhenius behaviour.

In [ ]:
if all_peaks_df_list:
    df_all_peaks = pd.concat(all_peaks_df_list, ignore_index=True)

    # skip Brouwer if pO2 data is not available (CSV/TXT entry mode)
    _has_pO2 = (
        "pO2_mean" in df_all_peaks.columns
        and df_all_peaks["pO2_mean"].notna().any()
    )
    if not _has_pO2:
        print("[SKIP] Brouwer diagram: no pO2 data available. "
              "pO2 is recorded only when using furnace log data (stage 0 + stage 1).")
    else:
        peak1_data = df_all_peaks[df_all_peaks["peak_id"] == BROUWER_PEAK_ID]
        n_cond_p1  = peak1_data["condition"].nunique() if not peak1_data.empty else 0
        print(f"Peak {BROUWER_PEAK_ID} data found in {n_cond_p1} condition(s) of {len(conditions)} total.")

        if n_cond_p1 < 2:
            print("  [SKIP] Brouwer diagram requires 2 or more conditions. "
                  "Run Stage 3 on more atmospheric conditions first.")
        else:
            brouwer_dir = RESULTS_BASE / "pO2"
            # Every peak gets its official diagram, all with the same
            # BROUWER_TEMPS filter: stale per-peak figures from older runs
            # would otherwise survive on disk and mix analysis vintages.
            for _pid in sorted(int(p) for p in df_all_peaks["peak_id"].unique()):
                _n_cond = df_all_peaks[df_all_peaks["peak_id"] == _pid]["condition"].nunique()
                if _n_cond < 2:
                    print(f"  [SKIP] Peak {_pid}: present in {_n_cond} condition(s)")
                    continue
                fig_brouwer = plot_brouwer(
                    df_all        = df_all_peaks,
                    save_dir      = brouwer_dir,
                    sample_name   = sample_id,
                    peak_id       = _pid,
                    temps_to_plot = BROUWER_TEMPS,
                    add_slopes    = True,
                    slopes        = tuple(BROUWER_SLOPES),
                )
                if _pid == BROUWER_PEAK_ID:
                    # raw Figure (not pyplot): shown explicitly, nothing to close
                    display(fig_brouwer)
            print(f"Brouwer diagrams saved in {brouwer_dir.relative_to(NOTEBOOK_DIR)}")

            print(f"\nPeak {BROUWER_PEAK_ID} data used for Brouwer diagram:")
            tbl = (
                peak1_data[["condition", "T_nominal", "pO2_mean", "R_i", "sigma_Sm_i"]]
                .sort_values(["T_nominal", "pO2_mean"])
                .reset_index(drop=True)
            )
            tbl["lg_pO2"]   = tbl["pO2_mean"].apply(
                lambda x: f"{np.log10(x):.3f}" if x > 0 else "N/A")
            tbl["lg_sigma"] = tbl["sigma_Sm_i"].apply(
                lambda x: f"{np.log10(x/100):.3f}" if x > 0 else "N/A")
            display(tbl[["condition", "T_nominal", "lg_pO2", "lg_sigma", "R_i"]])
else:
    print("No conditions processed; nothing to aggregate.")

## Step 2b: Brouwer: select peak and temperatures

Select a **peak** and (optionally) a **temperature** subset, then press
**↻ Replot Brouwer**. The figure `Brouwer_Peak{N}_{sample}.{png,pdf}` is saved to `Results/pO2/`.
Requires Step 2 to have been run first (uses `df_all_peaks` in memory).

In [ ]:
# Brouwer selector: replot the p(O2) diagram for a chosen peak / temperatures /
# conditions from df_all_peaks (Step 2, in memory); never recomputes fits.
# Overwrites the canonical Brouwer_Peak{N}_{sample}.{png,pdf}.
# Widget cleared synchronously via out_br.outputs = () (clear_output(wait=True) is
# lazy in some frontends and doubled the output); plot_brouwer returns a raw Figure,
# so the inline backend cannot re-render it on its own.
_HAS_WIDGETS_BR = _HAS_WIDGETS

if _HAS_WIDGETS_BR and ("df_all_peaks" in dir()) and not df_all_peaks.empty:
    _brouwer_dir = RESULTS_BASE / "pO2"
    _peak_ids  = sorted(int(p) for p in df_all_peaks["peak_id"].unique())
    _all_temps = sorted(int(t) for t in df_all_peaks["T_nominal"].unique())
    _all_conds = sorted(df_all_peaks["condition"].unique())

    w_peak = W.Dropdown(options=_peak_ids,
                        value=BROUWER_PEAK_ID if BROUWER_PEAK_ID in _peak_ids else _peak_ids[0],
                        description="Peak:", layout=W.Layout(width="180px"))
    w_temps = W.SelectMultiple(options=_all_temps, value=tuple(_all_temps),
                               description="T [°C]:", rows=min(9, len(_all_temps)),
                               layout=W.Layout(width="180px"))
    # Condition selector; deselect e.g. the Ar condition to drop it from the diagram.
    w_conds = W.SelectMultiple(options=_all_conds, value=tuple(_all_conds),
                               description="Cond:", rows=min(6, len(_all_conds)),
                               layout=W.Layout(width="440px"))
    _slope_opts = ["-1/4", "-1/6", "0", "+1/6", "+1/4"]
    w_slopes = W.SelectMultiple(options=_slope_opts,
                                value=tuple(s for s in BROUWER_SLOPES if s in _slope_opts),
                                description="Slopes:", rows=5,
                                layout=W.Layout(width="180px"))
    w_br_go = W.Button(description="↻ Replot Brouwer", button_style="primary",
                       layout=W.Layout(width="200px"))
    out_br = W.Output()

    def _brouwer_impl():
        peak_id = int(w_peak.value)
        sel_T   = [int(t) for t in w_temps.value]
        temps   = sel_T if (sel_T and len(sel_T) != len(_all_temps)) else None
        sel_C   = list(w_conds.value) or _all_conds
        df_sel  = df_all_peaks[df_all_peaks["condition"].isin(sel_C)]
        sub     = df_sel[df_sel["peak_id"] == peak_id]
        n_cond  = sub["condition"].nunique() if not sub.empty else 0
        out_br.outputs = ()   # synchronous clear: previous replot is gone NOW
        with out_br:
            if n_cond < 2:
                print(f"[SKIP] Peak {peak_id} present in only {n_cond} selected condition(s); "
                      "Brouwer needs >= 2. Select more conditions.")
                return
            fig = plot_brouwer(
                df_all=df_sel, save_dir=_brouwer_dir,
                sample_name=sample_id, peak_id=peak_id,
                temps_to_plot=temps,
                slopes=tuple(w_slopes.value),
            )
            _display(fig)
            print(f"Saved -> {(_brouwer_dir / f'Brouwer_Peak{peak_id}_{sample_id}')}.png/.pdf"
                  "  (overwrites the Step-2 figure)")
            print(f"  peak={peak_id}  temps={'all' if temps is None else temps}")
            print(f"  conditions ({n_cond}): {sel_C}")
    _br_busy = [False]

    def _on_brouwer(_btn):
        # ignore clicks queued while a replot is already running
        if _br_busy[0]:
            return
        _br_busy[0] = True
        w_br_go.disabled = True
        try:
            _brouwer_impl()
        finally:
            _br_busy[0] = False
            w_br_go.disabled = False
    w_br_go.on_click(_on_brouwer)

    _display(W.VBox([W.HBox([w_peak, w_temps, w_conds, w_slopes]), w_br_go, out_br]))
elif _HAS_WIDGETS_BR:
    print("[INFO] df_all_peaks not available; run Step 2 first.")

## Step 3: Ionic/electronic decomposition (transference numbers)

Per-temperature Patterson fit of the Brouwer data:

```
sigma(pO2) = sigma_ion + sigma_p * pO2^(+x) + sigma_n * pO2^(-x)     x = TRANSF_EXPONENT
```

Linear in the three coefficients, solved with non-negative least squares
(sigma_i >= 0). The ionic transference number follows at every pO2:
`t_ion = sigma_ion / sigma_tot`, `t_el = 1 - t_ion` (the local Brouwer slope
equals `x * t_el`: plateau = purely ionic, +x slope = purely electronic).
The table is exported and one figure is drawn for **every** peak.


In [ ]:
# Transference numbers: table for all peaks, figure for BROUWER_PEAK_ID
from pipeline.plots import (fit_transference, plot_brouwer_transference,
                            plot_transference_arrhenius)

if "df_all_peaks" not in globals():
    print("[SKIP] transference: run Step 1 and Step 2 first (df_all_peaks not in memory)")
elif not ("pO2_mean" in df_all_peaks.columns and df_all_peaks["pO2_mean"].notna().any()):
    print("[SKIP] transference: no pO2 data available")
else:
    brouwer_dir = RESULTS_BASE / "pO2"
    _t_tables = []
    for _pid in sorted(df_all_peaks["peak_id"].unique()):
        _df_t = fit_transference(df_all_peaks, peak_id=int(_pid),
                                 exponent=TRANSF_EXPONENT, temps=BROUWER_TEMPS)
        if not _df_t.empty:
            _t_tables.append(_df_t)
    if not _t_tables:
        print("[SKIP] transference: not enough pO2 points per temperature")
    else:
        df_transference = pd.concat(_t_tables, ignore_index=True)
        _xlsx = brouwer_dir / "stage4_transference.xlsx"
        try:
            with pd.ExcelWriter(_xlsx, engine="openpyxl") as _w:
                df_transference.to_excel(_w, sheet_name="Transference", index=False)
            print(f"Transference table saved: {_xlsx.relative_to(NOTEBOOK_DIR)}")
        except Exception as _exc:
            print(f"[WARN] could not write {_xlsx.name}: {type(_exc).__name__}: {_exc}")

        # one figure per process; figures restricted to TRANSF_PEAK_IDS
        _fig_pids = (sorted(df_transference["peak_id"].unique())
                     if TRANSF_PEAK_IDS is None else list(TRANSF_PEAK_IDS))
        for _pid in _fig_pids:
            print(f"Peak {_pid}:")
            try:
                fig_tr = plot_brouwer_transference(
                    df_all_peaks, save_dir=brouwer_dir, sample_name=sample_id,
                    peak_id=int(_pid), exponent=TRANSF_EXPONENT,
                    temps_to_plot=BROUWER_TEMPS)
                plt.show()
                plt.close(fig_tr)
            except ValueError as _exc:
                print(f"  [SKIP] {_exc}")
            # Arrhenius of the partial conductivities: the rigorous check
            # that sigma_ion and sigma_p are two distinct activated channels
            fig_ta = plot_transference_arrhenius(
                df_transference, save_dir=brouwer_dir,
                sample_name=sample_id, peak_id=int(_pid))
            if fig_ta is not None:
                plt.show()
                plt.close(fig_ta)

        # one row per (peak, T): fitted components and the t_ion range over pO2
        _sum = (df_transference
                .groupby(["peak_id", "T_nominal"])
                .agg(sigma_ion=("sigma_ion", "first"), sigma_p=("sigma_p", "first"),
                     sigma_n=("sigma_n", "first"), R2=("R2", "first"),
                     t_ion_min=("t_ion", "min"), t_ion_max=("t_ion", "max"))
                .reset_index())
        display(_sum.round(4))


## Output summary

All figures have been exported to the sub-folders of `Results/{condition}/`:
- `DRT/`          : DRT γ(τ) stacked per temperature
- `Nyquist-Bode/` : Nyquist overlay + Bode
- `Arrhenius/`    : Arrhenius 2×2 panel
- `pO2/`          : Brouwer p(O₂) diagram (multi-condition)

Each figure is saved as **PNG** (quick preview) and **PDF** (publication ready).

**Next step:** run [stage5_model.ipynb](stage5_model.ipynb)